In [25]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import pandas as pd

In [ ]:
class Bert(nn.Module):
    def __init__(self, bert_base_model_dir, label_size, drop_out_rate = 0.5):
        super(Bert,self).__init__()
        self.label_size = label_size

        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_base_model_dir) 
        self.bert_model = BertModel.from_pretrained(bert_base_model_dir)

        self.dropout=nn.Dropout(drop_out_rate)
        self.cls_layer = nn.Linear(self.bert_model.config.hidden_size, label_size)

    def forward(self, input_ids, attention_mask, token_type_ids=None, position_ids=None, head_mask=None, labels=None):
        last_hidden_state, pooled_output = self.bert_model(
            input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, 
            position_ids=position_ids, head_mask=head_mask, inputs_embeds=None, return_dict=False
        )
        cls_outs = self.dropout(pooled_output)
        logits = self.cls_layer(cls_outs)

        best_labels = torch.argmax(logits, dim=-1)


        if labels is not None:
            loss = CrossEntropyLoss(ignore_index=-1)(logits, labels)
            return best_labels, loss
        
        return best_labels
    def get_bert_tokenizer(self):
        return self.bert_tokenizer

In [18]:
label_size = 6
model_path = "./bert-base-chinese"
batch_size = 32
device = torch.accelerator.current_accelerator().type

In [27]:
best_model = Bert(model_path, label_size).to(device)
best_model.load_state_dict(torch.load("./trained_model/trained_bert_epoch_4.pth"))
tokenizer = BertTokenizer.from_pretrained(model_path)

In [31]:
def tokenize(texts):
    return tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )


In [20]:
df = pd.read_excel('./data/xuemeng_for_res/cleaned_chat_log.xlsx', sheet_name=0)
pred_texts = df[4].astype(str).tolist()
pred_texts_size = len(pred_texts)

classes = ['happy', 'angry', 'sad', 'fear', 'surprise', 'neutral']
print(df.head())

   0                    1               2        3           4
0  1  2025-09-09 15:16:54   自爆法蒂瑪要od花刀雙子塔   type_2  艾特我看小猫绝美侧颜
1  2  2025-09-09 15:16:54   自爆法蒂瑪要od花刀雙子塔   type_2  艾特我看小猫绝美侧颜
2  3  2025-09-09 15:20:13            桂仁蒸叶   type_2   多pre一下就好了
3  4  2025-09-09 15:20:14  Waldeinsamkeit   type_2     口语一点都不会
4  6  2025-09-09 15:20:16  Waldeinsamkeit  type_17      [[大哭]]


In [ ]:
all_preds = []

best_model.eval()
with torch.no_grad():
    for i in range(0, pred_texts_size, batch_size):
        batch_texts = pred_texts[i : i + batch_size]
       
        encoded =  tokenize(batch_texts)

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

            
        preds = best_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        all_preds.extend(preds.cpu().tolist())
    pred_classes = [classes[id] for id in all_preds]

    df[5] = pred_classes
    save_path = './data/xuemeng_for_res/predicted_chat_log.xlsx'
    df.to_excel(save_path, index=False)

print("Done!")
print(df.head(10))




Done!
    0                    1               2        3                    4  \
0   1  2025-09-09 15:16:54   自爆法蒂瑪要od花刀雙子塔   type_2           艾特我看小猫绝美侧颜   
1   2  2025-09-09 15:16:54   自爆法蒂瑪要od花刀雙子塔   type_2           艾特我看小猫绝美侧颜   
2   3  2025-09-09 15:20:13            桂仁蒸叶   type_2            多pre一下就好了   
3   4  2025-09-09 15:20:14  Waldeinsamkeit   type_2              口语一点都不会   
4   6  2025-09-09 15:20:16  Waldeinsamkeit  type_17               [[大哭]]   
5   7  2025-09-09 15:20:29            桂仁蒸叶   type_2               大家一样紧张   
6   8  2025-09-09 15:20:33            桂仁蒸叶   type_2               没什么好怕的   
7   9  2025-09-09 15:20:35      1146348679   type_2  选课本本里的新生研讨课是啥呀[/睁眼]   
8  11  2025-09-09 15:20:40            桂仁蒸叶   type_2                   水课   
9  12  2025-09-09 15:20:41  Summer_Forever   type_2             pre是什么……   

          5  
0     happy  
1     happy  
2       sad  
3     angry  
4       sad  
5      fear  
6     happy  
7  surprise  
8       sad  
9   neutral  
